# Финальный воспроизводимый пакет SBMTeam

**Команда Kaggle:** SBMTeam  
**Участники:** Peretiatko Vsevolod (`sevaperetiatko`) и Platon Usachev (`omnistudent`)  
**Основной submission:** `submission_NEW5_sqrt_s100_aux20.csv`  
**Дополнительный финальный вариант:** `submission_rescue_corrected_best.csv`

В этом ноутбуке оставлен только воспроизводимый финальный конвейер: подготовка признаков, обучение двух финальных ансамблей, несколько содержательных базовых моделей для сравнения, расчёт validation ROC-AUC и сохранение файлов предсказаний. Поисковые эксперименты и Optuna-подбор сюда не перенесены, чтобы запуск укладывался в ограничения финального пакета.


## Условия воспроизводимости

- Все пути относительны папки пакета; интернет и скрытые локальные файлы не используются.
- Фиксируется единый seed для Python, NumPy и всех стохастических моделей.
- Разбиение train/validation выполняется один раз со стратификацией.
- Обучаемые преобразования признаков строятся без утечки: признаки обучающей части формируются out-of-fold, а validation и test только преобразуются объектами, обученными на train.
- Дерево решений, случайный лес и градиентный бустинг получают расширенный набор engineered-признаков. Простой RBF SVC намеренно использует только исходные 512 признаков и служит понятной точкой отсчёта.
- Validation используется только для оценки готовых конфигураций и не участвует в обучении преобразований или моделей.
- После обучения заново создаются `final_submission.csv`, `submission_NEW5_sqrt_s100_aux20.csv` и `submission_rescue_corrected_best.csv`.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from final_model_runner import MODEL_CONFIGURATIONS, SEED, run

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 240)

print("Зафиксированный seed:", SEED)
print("Рабочая папка:", Path.cwd())
print("Доступных CPU-ядер:", os.cpu_count())


## Зафиксированные конфигурации моделей

Ниже перечислены десять сравниваемых решений. Первые четыре дают контраст между классическими деревьями и простым SVC, следующие четыре показывают вклад отдельных SVC-компонентов, а последние два являются финальными ансамблями. Параметры уже зафиксированы результатами предыдущих экспериментов и при финальном запуске не подбираются повторно.

Текст в ячейках переносится, а ограничение длины столбцов отключено, поэтому таблицу можно читать полностью и при необходимости прокручивать по горизонтали.


In [ ]:
configuration_table = pd.DataFrame(MODEL_CONFIGURATIONS).rename(columns={
    "model": "Модель",
    "role": "Назначение",
    "parameters": "Параметры",
    "feature_set": "Признаки",
})

display(
    configuration_table.style
    .set_properties(**{
        "white-space": "pre-wrap",
        "text-align": "left",
        "vertical-align": "top",
    })
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left"), ("white-space", "nowrap")]},
        {"selector": "table", "props": [("width", "100%")]},
    ])
    .hide(axis="index")
)


### Как устроены два финальных ансамбля

Оба результата являются взвешенными смесями рангов предсказаний. Такой подход делает компоненты сопоставимыми по масштабу и устойчивее объединяет модели с разными распределениями score. Веса были выбраны на предыдущем исследовательском этапе и здесь остаются фиксированными.


In [ ]:
FINAL_ENSEMBLES = {
    "NEW5 (финальная)": {
        "Ранг S100 SVC": 0.80,
        "Ранг clean_aux SVC": 0.20,
    },
    "Rescue T6": {
        "Ранг corrected K0 SVC": 0.55,
        "Ранг fine6 SVC": 0.25,
        "Ранг S100 SVC": 0.20,
    },
}

display(
    pd.DataFrame(FINAL_ENSEMBLES).fillna(0.0)
    .rename_axis("Компонент")
    .style.format("{:.2f}")
)


## Обучение, оценка и генерация submission

Один вызов `run()` выполняет весь конвейер. Он загружает конкурсные CSV из относительных путей, создаёт leakage-safe признаки, обучает компоненты двух финальных ансамблей и десять моделей из таблицы сравнения, считает validation ROC-AUC, затем сохраняет вероятности для test.

Древесные модели используют исходные признаки вместе с геометрическими, плотностными, агрегатными, координатными, kNN- и PCA-блоками. Простой SVC использует только raw-признаки; это позволяет увидеть вклад разработанного feature engineering. Время каждой модели измеряется отдельно, а служебные затраты конвейера приводятся дополнительно.


In [ ]:
result = run(output_dir=Path("."))

print("Созданы файлы:")
for output_path in result["output_paths"]:
    print(" -", output_path)


## Итоговое сравнение десяти моделей

В компактной итоговой таблице оставлены только четыре требуемых поля: название модели, validation ROC-AUC, время выполнения и seed. Подробные параметры, назначение и набор признаков уже приведены выше в таблице конфигураций, поэтому здесь они не дублируются.


In [ ]:
experiment_table = result["experiment_table"].copy()

display(
    experiment_table.style
    .format({
        "Validation ROC-AUC": "{:.6f}",
        "Время, сек": "{:.2f}",
    })
    .hide(axis="index")
)

timing_names = {
    "preprocessing_seconds": "Подготовка данных и признаков",
    "final_components_seconds": "Компоненты финальных ансамблей",
    "comparison_models_seconds": "Сравнительные модели",
    "test_prediction_seconds": "Формирование test-предсказаний после fit",
    "total_seconds": "Полное время конвейера",
}
timing_table = pd.DataFrame([
    {"Этап": timing_names.get(name, name), "Время, сек": seconds}
    for name, seconds in result["timings"].items()
])
display(timing_table.style.format({"Время, сек": "{:.2f}"}).hide(axis="index"))


## Версии среды

Таблица фиксирует версии Python и основных библиотек, от которых зависит воспроизводимость расчётов.


In [ ]:
versions_table = pd.DataFrame(
    [{"Пакет": name, "Версия": version} for name, version in result["versions"].items()]
)
display(versions_table.style.hide(axis="index"))


## Автоматические проверки результата

В конце проверяется, что `row_id` совпадает с test в исходном порядке, вероятности конечны и находятся в диапазоне `[0, 1]`, а все три ожидаемых CSV действительно созданы. Эти проверки ловят наиболее опасные ошибки сборки submission до загрузки на Kaggle или Moodle.


In [ ]:
assert result["row_id_exact"], "row_id в submission не совпадает с test.csv"
assert result["probabilities_valid"], "Предсказания содержат некорректные вероятности"
assert all(Path(path).is_file() for path in result["output_paths"]), "Созданы не все CSV-файлы"

print("Проверка row_id: пройдена")
print("Проверка вероятностей: пройдена")
print("Все ожидаемые CSV созданы: да")
